In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from typing import Optional
import torch.nn.functional as F
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LRScheduler
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from dataset import Multi30kDataset
from lr_scheduler import NoamScheduler
from model import Transformer, make_src_mask, make_tgt_mask

from tqdm import tqdm

from nltk.translate.bleu_score import corpus_bleu
from dataset import *
from train import *
import wandb

In [2]:
language_dataset = Multi30kDataset(split = "test")
processed_dataset = language_dataset.process_data()

dataset_obj = TranslationDataset(processed_dataset)
dataloader = DataLoader(
    dataset_obj,
    batch_size=5,
    shuffle=True,
    collate_fn=collate_fn
)

Building vocab, please wait...


100%|██████████| 1000/1000 [00:00<00:00, 45549.66it/s]


In [24]:
sample = next(iter(dataloader))

print(' '.join([language_dataset.de_itos[i.item()] for i in sample[0][0]]))
print(' '.join([language_dataset.en_itos[i.item()] for i in sample[1][0]]))

<sos> kinder bemühen sich, beim seilziehen zu gewinnen. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>
<sos> children fight to win a tug-of-war battle. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>


In [26]:
src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)

In [28]:
transformer.infer("<sos> kinder bemühen sich, beim seilziehen zu gewinnen. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>")

Building vocab, please wait...


Processing test: 100%|██████████| 1000/1000 [00:00<00:00, 47169.94it/s]


'<sos> children are playing tennis. <eos>'

In [ ]:
sos_idx = 2
eos_idx = 3
device = "cpu"

src, tgt = next(iter(dataloader))
# sample_src = src[0].unsqueeze(0)
# sample_tgt = tgt[0].unsqueeze(0)
src_mask = make_src_mask(src).to(device)


src_vocab = len(language_dataset.de_vocab)
tgt_vocab = len(language_dataset.en_vocab)

transformer = Transformer(src_vocab_size = src_vocab, tgt_vocab_size = tgt_vocab,
                          d_model = 512, N = 6, num_heads = 8, d_ff = 2048,
                          dropout = 0.1)


load_checkpoint("checkpoint.pt", transformer)

9

In [ ]:
text = "der vieler"
torch.tensor([language_dataset.de_vocab[i] for i in text.split()]).unsqueeze(0)

torch.Size([1, 2])

In [9]:
src.shape

torch.Size([5, 12])

In [15]:
prediction_idx = greedy_decode(transformer, src, src_mask, 40, sos_idx, eos_idx, "cpu", break_at_eos = True)

In [27]:
no_clean = ' '.join([language_dataset.en_itos[i.item()] for i in prediction_idx[0]])
no_clean.split("<sos>")[-1].strip().split("<eos>")[0].strip()

'three boys are doing different arts. in a field.'

In [16]:
for j in range(5):
    print("Source: ")
    for i in src[j]:
        print(language_dataset.de_itos[i.item()], end = " ")

    print("\n\nTarget, ground truth: ")
    for i in tgt[j]:
        print(language_dataset.en_itos[i.item()], end = " ")

    print("\n\nTarget, prediction: ")
    for i in prediction_idx[j]:
        print(language_dataset.en_itos[i.item()], end = " ")
    
    print("\n-----\n")

Source: 
<sos> ein junge in einer rot-weißen badehose springt rückwärts in einen schönen pool. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 

Target, ground truth: 
<sos> a boy wearing red and white swimming trunks diving backwards in a beautiful pool. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 

Target, prediction: 
<sos> a boy in a red and white swim jacket is jumping into a swimming pool. <eos> <eos> pool. <eos> pool. <eos> <eos> <eos> pool. <eos> pool. <eos> 
-----

Source: 
<sos> ein kleiner junge in baseballmontur holt mit einem schläger hinter seinem kopf in richtung eines vor ihm montierten baseballs aus. <eos> <pad> <pad> <pad> <pad> <pad> 

Target, ground truth: 
<sos> a small boy wearing baseball regalia holds a bat behind his head with a baseball mounted in front of him. <eos> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> 

Target, prediction: 
<sos> a

In [ ]:
bleu_score = evaluate_bleu(transformer, dataloader, language_dataset.en_itos)
bleu_score